# Real-World Data (RWD) Medallion Pipeline for Diabetic Patient Readmission Risk

## Project overview

This notebook demonstrates a **Bronze → Silver → Gold medallion data pipeline** for a small mock Electronic Health Record (EHR) dataset.

The pipeline shows how raw JSON clinical data can be:

1. **Ingested** without changing the original payload (**Bronze**)
2. **Parsed, cleaned, standardized, and transformed** into an analytics-ready structure (**Silver**)
3. **Aggregated into clinical metrics** for reporting (**Gold**)

> **Important:** This is an educational data-engineering demonstration using synthetic/mock patient records. It is **not a clinical prediction model** and must not be used for medical decision-making.

### Technology

- Python
- Apache Spark / PySpark
- Delta Lake
- JSON parsing
- OMOP-style concept standardization
- Databricks-compatible `saveAsTable()` workflow


## 1. Medallion architecture

```text
Mock EHR JSON
     │
     ▼
┌───────────────┐
│ BRONZE        │  Raw JSON + ingestion timestamp
│ bronze_ehr_raw│
└───────┬───────┘
        │ Parse JSON + validate
        ▼
┌──────────────────────┐
│ SILVER               │  Structured patient records
│ silver_patient_omop  │  + ICD-10 → OMOP-style concept ID
└──────────┬───────────┘
           │ Aggregate
           ▼
┌──────────────────────┐
│ GOLD                 │  Patient counts + readmission metrics
│ gold_diabetic_metrics│
└──────────────────────┘
```

### Why the three layers?

| Layer | Purpose | Example |
|---|---|---|
| Bronze | Preserve source data for traceability | Raw JSON payload |
| Silver | Clean and standardize data | Patient ID, diagnosis code, readmission flag |
| Gold | Produce business/clinical analytics | Readmission rate by concept |

This separation makes the pipeline easier to audit, debug, extend, and reuse.


## 2. Environment setup

This notebook is designed for a **Databricks/Spark environment with Delta Lake support**.

If `spark` is already provided by Databricks, no additional Spark session is required.

For a local PySpark environment, the following cell can be used to create a Spark session. Delta Lake configuration may require additional environment-specific setup.


In [ ]:
# Databricks normally provides the `spark` session automatically.
# Run this only if you are working in a compatible local PySpark environment.

from pyspark.sql import SparkSession

# Uncomment when required:
# spark = (
#     SparkSession.builder
#     .appName("RWD-Medallion-Diabetic-Readmission")
#     .getOrCreate()
# )

print("Spark session ready:", spark.version)


## 3. Import PySpark functions and define the mock source data

The source records simulate a simplified EHR event represented as JSON.

Each record contains:

- `patient_id` — source patient identifier
- `gender` — patient gender value
- `dob` — date of birth
- `diagnoses` — diagnosis code and description
- `readmitted_30d` — illustrative 30-day readmission indicator
- `ingestion_time` — time at which the record entered the pipeline

The data is intentionally small so that the transformations are easy to understand.


In [ ]:
from pyspark.sql.functions import (
    col,
    from_json,
    to_timestamp,
    expr,
    when,
    count,
    sum as spark_sum,
    round as spark_round
)
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    ArrayType
)

# Synthetic/mock EHR records
raw_data = [
    (
        '{"patient_id": "P-101", "gender": "F", "dob": "1968-05-14", '
        '"diagnoses": [{"code": "E11.9", "desc": "Type 2 Diabetes"}], '
        '"readmitted_30d": 1}',
        "2026-09-20 08:00:00"
    ),
    (
        '{"patient_id": "P-102", "gender": "M", "dob": "1975-11-22", '
        '"diagnoses": [{"code": "I10", "desc": "Hypertension"}], '
        '"readmitted_30d": 0}',
        "2026-09-20 08:05:00"
    ),
    (
        '{"patient_id": "P-103", "gender": "F", "dob": "1954-01-30", '
        '"diagnoses": [{"code": "E11.65", "desc": "Type 2 Diabetes with Hyperglycemia"}], '
        '"readmitted_30d": 1}',
        "2026-09-20 08:10:00"
    ),
    (
        '{"patient_id": "P-104", "gender": "M", "dob": "1982-08-03", '
        '"diagnoses": [{"code": "E11.9", "desc": "Type 2 Diabetes"}], '
        '"readmitted_30d": 0}',
        "2026-09-20 08:15:00"
    )
]

bronze_df = spark.createDataFrame(
    raw_data,
    ["raw_json_payload", "ingestion_time"]
)

bronze_df.show(truncate=False)


## 4. Bronze layer — raw ingestion

The Bronze layer keeps the source JSON payload intact.

### Design principle

**Do not over-transform raw data at ingestion time.**

Keeping the original payload helps with:

- traceability
- replaying transformations
- debugging
- auditing
- investigating data-quality issues

The ingestion timestamp records when each record entered the pipeline.


In [ ]:
# Write the raw source data to a Delta table.
# This requires a Spark/Databricks environment with Delta Lake support.

bronze_df.write     .format("delta")     .mode("overwrite")     .saveAsTable("bronze_ehr_raw")

print("Bronze table created: bronze_ehr_raw")


## 5. Silver layer — schema enforcement and clinical standardization

The Silver layer converts semi-structured JSON into a structured table.

### Transformation steps

1. Define an explicit JSON schema.
2. Parse the raw JSON with `from_json()`.
3. Extract patient and diagnosis fields.
4. Convert the date of birth into a timestamp.
5. Create an ICD-10 diagnosis field.
6. Map selected ICD-10 codes to an OMOP-style concept ID.
7. Remove records without a patient identifier.
8. Remove duplicate patient records.

### Important assumption

The source contains one diagnosis array per record. For simplicity, this demonstration uses the **first diagnosis** with `parsed.diagnoses[0]`.

In a production pipeline, all diagnosis entries would normally be normalized rather than discarding additional diagnoses.


In [ ]:
# JSON schema for the incoming EHR payload
ehr_schema = StructType([
    StructField("patient_id", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("dob", StringType(), True),
    StructField("readmitted_30d", IntegerType(), True),
    StructField(
        "diagnoses",
        ArrayType(
            StructType([
                StructField("code", StringType(), True),
                StructField("desc", StringType(), True)
            ])
        ),
        True
    )
])

# Bronze → Silver
silver_df = (
    spark.table("bronze_ehr_raw")
    .withColumn(
        "parsed",
        from_json(col("raw_json_payload"), ehr_schema)
    )
    .select(
        col("parsed.patient_id").alias("person_source_value"),
        col("parsed.gender").alias("gender_source_value"),
        to_timestamp(
            col("parsed.dob"),
            "yyyy-MM-dd"
        ).alias("birth_datetime"),
        expr("parsed.diagnoses[0].code").alias("icd10_code"),
        col("parsed.readmitted_30d").alias("is_30day_readmitted"),
        col("ingestion_time")
    )
    # Demonstration mapping to OMOP-style concept IDs.
    # The mapping is intentionally simplified for this educational example.
    .withColumn(
        "omop_concept_id",
        when(col("icd10_code").startswith("E11"), 201826)
        .otherwise(320128)
    )
    .filter(col("person_source_value").isNotNull())
    .dropDuplicates(["person_source_value"])
)

silver_df.show(truncate=False)


### Silver-layer data dictionary

| Column | Meaning |
|---|---|
| `person_source_value` | Source patient identifier |
| `gender_source_value` | Source gender value |
| `birth_datetime` | Parsed date of birth |
| `icd10_code` | First diagnosis ICD-10 code |
| `is_30day_readmitted` | Illustrative 30-day readmission flag |
| `ingestion_time` | Bronze ingestion timestamp |
| `omop_concept_id` | Simplified standardized concept identifier |

The OMOP mapping in this notebook is a **demonstration mapping**, not a complete clinical terminology mapping service.


In [ ]:
# Persist the structured Silver layer
silver_df.write     .format("delta")     .mode("overwrite")     .saveAsTable("silver_patient_omop")

print("Silver table created: silver_patient_omop")


## 6. Gold layer — analytical readmission metrics

The Gold layer is designed for downstream analytics and reporting.

For each standardized concept, we calculate:

- **Total patients**
- **Total 30-day readmissions**
- **Readmission rate (%)**

Formula:

\[
Readmission\ Rate = \frac{Total\ Readmissions}{Total\ Patients} \times 100
\]

Because this notebook uses only four synthetic records, the resulting metrics are illustrative rather than statistically meaningful.


In [ ]:
# Silver → Gold
gold_df = (
    spark.table("silver_patient_omop")
    .groupBy("omop_concept_id")
    .agg(
        count("person_source_value").alias("total_patients"),
        spark_sum("is_30day_readmitted").alias("total_readmissions")
    )
    .withColumn(
        "readmission_rate_pct",
        spark_round(
            (col("total_readmissions") / col("total_patients")) * 100,
            2
        )
    )
    .orderBy(col("readmission_rate_pct").desc())
)

gold_df.show()


In [ ]:
# Persist the Gold analytics table
gold_df.write     .format("delta")     .mode("overwrite")     .saveAsTable("gold_diabetic_metrics")

print("Gold table created: gold_diabetic_metrics")


## 7. Expected result for the supplied mock records

From the four synthetic records:

- Three records have an `E11...` diagnosis and therefore use the demonstration concept ID `201826`.
- One record has `I10` and therefore uses the demonstration fallback concept ID `320128`.
- Two of the three diabetes-coded records have `readmitted_30d = 1`.

Therefore, the diabetes-coded group is expected to show:

**3 patients → 2 readmissions → 66.67% illustrative readmission rate**

The hypertension-coded group is expected to show:

**1 patient → 0 readmissions → 0.00% illustrative readmission rate**

These percentages should **not** be interpreted as a clinical risk estimate because the sample is synthetic and extremely small.


## 8. Data-quality and production considerations

This notebook intentionally keeps the pipeline simple. A production implementation should consider:

### Data quality
- Validate required fields.
- Check invalid dates and malformed JSON.
- Validate that `readmitted_30d` contains only permitted values.
- Track rejected/quarantined records.
- Add completeness and uniqueness checks.

### Clinical terminology
- Use an authoritative terminology service or validated OMOP vocabulary tables.
- Maintain versioned ICD-10 → OMOP mappings.
- Do not use a simple prefix/fallback mapping for production clinical analytics.

### Patient privacy
- Minimize personally identifiable information.
- Apply access controls and encryption.
- Follow applicable healthcare privacy and governance requirements.
- Use de-identified or synthetic data for public GitHub demonstrations.

### Pipeline engineering
- Prefer incremental ingestion for real workloads.
- Add data lineage and audit columns.
- Add automated tests.
- Partition/optimize tables based on actual workload patterns.
- Use schema evolution deliberately rather than implicitly.


## 9. Key learning outcomes

After completing this notebook, you should be able to explain:

1. What a **medallion architecture** is.
2. Why raw data is retained in a **Bronze layer**.
3. How PySpark parses nested JSON into structured columns.
4. Why data cleaning and standardization belong in the **Silver layer**.
5. How an analytical **Gold layer** can expose business/clinical metrics.
6. How Delta tables can persist each pipeline layer.
7. Why synthetic healthcare data is preferable for a public GitHub portfolio project.


## 10. Portfolio / GitHub project summary

### Project: Real-World Data (RWD) Medallion Pipeline for Diabetic Patient Readmission Risk

**Objective:** Demonstrate a healthcare data-engineering pipeline that transforms semi-structured EHR JSON into standardized and analytics-ready clinical metrics.

**Architecture:** Bronze → Silver → Gold

**Tools:** Python, PySpark, Spark SQL functions, Delta Lake, Databricks-compatible tables

**Core skills demonstrated:**
- Data ingestion
- JSON schema design
- Data transformation
- Data cleaning
- Clinical-code standardization
- Aggregation
- Delta Lake
- Medallion architecture
- Healthcare data engineering concepts

### Suggested GitHub repository structure

```text
rwd-medallion-diabetic-readmission/
│
├── notebooks/
│   └── RWD_Medallion_Pipeline_Diabetic_Readmission_Risk.ipynb
│
├── README.md
├── requirements.txt
└── LICENSE
```

This notebook is suitable as a portfolio demonstration of **PySpark + healthcare data engineering + medallion architecture**.
